# Task 2: Aspect Sentiment Classification (ASC)
## Baselines: TextCNN, RNN, LSTM, GRU, BiLSTM, BiGRU

ASC = Classification: Cho cau va vi tri aspect, phan loai POSITIVE / NEGATIVE / NEUTRAL.


## 1. Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from IPython.display import display

from src.utils.preprocess import load_preprocessed
from src.asc.asc_dataset import ASCDataset
from src.asc.asc_model import build_asc_model
from src.utils.engine import train_asc_model, predict_asc
from src.utils.metrics import evaluate_asc
from src.utils.visualization import plot_training_curves, plot_model_comparison, plot_confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42); np.random.seed(42)
print(f"Device: {device}")


## 2. Load Preprocessed Data
Vocab va Embeddings da duoc save tu Notebook 01. Khong can train lai.


In [ ]:
PREP_DIR = os.path.join("..", "..", "preprocessed")
word2idx, emb_matrix, train_items, dev_items, test_items = load_preprocessed(PREP_DIR)
VOCAB_SIZE = len(word2idx)
EMB_DIM = emb_matrix.shape[1]
print(f"Loaded! Vocab: {VOCAB_SIZE} | Emb: {EMB_DIM}d")


## 3. DataLoaders


In [ ]:
MAX_LEN = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3
BATCH_SIZE = 64
EPOCHS = 30
PATIENCE = 7

train_ds = ASCDataset(train_items, word2idx, MAX_LEN)
dev_ds = ASCDataset(dev_items, word2idx, MAX_LEN)
test_ds = ASCDataset(test_items, word2idx, MAX_LEN)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, BATCH_SIZE)
test_loader = DataLoader(test_ds, BATCH_SIZE)
print(f"ASC Samples - Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}")
print("(Moi aspect trong 1 cau = 1 sample rieng biet)")


## 4. Train All Baselines


In [ ]:
MODEL_TYPES = ["TextCNN", "RNN", "LSTM", "GRU", "BiLSTM", "BiGRU"]

all_models = {}
all_histories = {}

for model_type in MODEL_TYPES:
    model = build_asc_model(
        model_type=model_type, vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM,
        hidden_dim=HIDDEN_DIM, num_classes=3,
        pretrained_emb=emb_matrix, n_layers=NUM_LAYERS, dropout=DROPOUT,
    ).to(device)

    model, history = train_asc_model(
        model, train_loader, dev_loader, device,
        lr=1e-3, epochs=EPOCHS, patience=PATIENCE, model_name=model_type
    )
    all_models[model_type] = model
    all_histories[model_type] = history


## 5. Evaluate on Test Set


In [ ]:
CLASS_NAMES = ["POSITIVE", "NEGATIVE", "NEUTRAL"]
all_results = []

for model_type, model in all_models.items():
    criterion = nn.CrossEntropyLoss()
    test_res = predict_asc(model, test_loader, criterion, device)
    m = evaluate_asc(test_res["labels"], test_res["preds"], CLASS_NAMES)

    all_results.append({
        "Model": model_type,
        "Accuracy": round(m["accuracy"], 4),
        "Macro_F1": round(m["macro_f1"], 4),
        "Weighted_F1": round(m["weighted_f1"], 4),
    })
    print(f"{model_type} | Acc: {m['accuracy']:.4f} | Macro F1: {m['macro_f1']:.4f}")

results_df = pd.DataFrame(all_results)
display(results_df.sort_values("Macro_F1", ascending=False).style.highlight_max(
    subset=["Macro_F1"], color="lightgreen"))


## 6. Visualization & Confusion Matrix


In [ ]:
plot_training_curves(all_histories)
plot_model_comparison(results_df, metric_col="Macro_F1", title="ASC: Macro F1 Comparison")

# Confusion Matrix of best model
best_name = results_df.loc[results_df["Macro_F1"].idxmax(), "Model"]
criterion = nn.CrossEntropyLoss()
best_test = predict_asc(all_models[best_name], test_loader, criterion, device)
best_m = evaluate_asc(best_test["labels"], best_test["preds"], CLASS_NAMES)
plot_confusion_matrix(best_m["confusion_matrix"], CLASS_NAMES, title=f"Confusion Matrix: {best_name}")


## 7. Save Results


In [ ]:
SAVE_DIR = os.path.join("..", "..", "results", "asc")
os.makedirs(SAVE_DIR, exist_ok=True)
results_df.to_csv(os.path.join(SAVE_DIR, "asc_results.csv"), index=False)
torch.save(all_models[best_name].state_dict(), os.path.join(SAVE_DIR, f"best_asc_{best_name}.pt"))
print(f"Best ASC: {best_name} (Macro F1={results_df['Macro_F1'].max():.4f}) -> Saved!")
